In [1]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [29]:
import os
import json
import random
import torch
from datetime import datetime
import numpy as np
import pandas as pd

In [3]:
# Canonical paths (match Notebook 08)
base_dir = "/content/drive/MyDrive/Final_Year_Project/Implementation/datasets/AntM2C_10M"
preprocessed_dir = f"{base_dir}/AntM2C_preprocessed"
checkpoints_dir = f"{base_dir}/checkpoints"
splits_dir = f"{preprocessed_dir}/splits"
split_features_dir = f"{preprocessed_dir}/split_features"

results_dir = f"{base_dir}/eval_results"
os.makedirs(results_dir, exist_ok=True)

In [5]:
# Artifact paths (Notebook 08 actual filenames)
model_ckpt_path = os.path.join(checkpoints_dir, "coldfusionx_model_epoch3.pt")
fusion_config_path = os.path.join(checkpoints_dir, "fusion_config.json")
meta_config_path = os.path.join(checkpoints_dir, "meta_config.json")
training_meta_path = os.path.join(checkpoints_dir, "training_meta_epoch3.json")

test_idx_path = os.path.join(splits_dir, "test_idx.npy")
val_idx_path = os.path.join(splits_dir, "val_idx.npy")

In [8]:
# Existence checks
required_paths = {
    "base_dir": base_dir,
    "preprocessed_dir": preprocessed_dir,
    "checkpoints_dir": checkpoints_dir,
    "splits_dir": splits_dir,
    "split_features_dir": split_features_dir,
    "results_dir": results_dir,
    "model_ckpt_path": model_ckpt_path,
    "fusion_config_path": fusion_config_path,
    "meta_config_path": meta_config_path,
    "training_meta_path": training_meta_path,
    "test_idx_path": test_idx_path,
}

In [9]:
missing = [k for k, p in required_paths.items() if not os.path.exists(p)]
if missing:
    print("Missing required paths:")
    for k in missing:
        print(f" - {k}: {required_paths[k]}")
    raise FileNotFoundError("One or more required files/folders are missing. Fix paths before continuing.")

In [11]:
# Reproducibility
SEED = 42           # consistent with Notebook 08
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

In [12]:
# Determinism flags (for evaluation reproducibility)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

In [13]:
# Load configs
with open(fusion_config_path, "r") as f:
    fusion_config = json.load(f)

with open(meta_config_path, "r") as f:
    meta_config = json.load(f)

with open(training_meta_path, "r") as f:
    training_meta = json.load(f)

In [14]:
# Load split indices
test_idx = np.load(test_idx_path)
val_idx = np.load(val_idx_path) if os.path.exists(val_idx_path) else None

In [16]:
# Extract key values (safe getters)
d_model = fusion_config.get("d_model", fusion_config.get("D_MODEL", None))
fewshot_n = meta_config.get("FEWSHOT_N", meta_config.get("fewshot_n", None))
num_scenes = meta_config.get("NUM_SCENES", meta_config.get("num_scenes", None))

batch_size = training_meta.get("batch_size", training_meta.get("BATCH_SIZE", None))

In [19]:
print("Key Config Values")
print("d_model:", d_model)
print("FEWSHOT_N:", fewshot_n)
print("num_scenes:", num_scenes)
print("batch_size (from training_meta):", batch_size)

print("\nSplit Sizes")
print("test_idx:", len(test_idx))
print("val_idx:", len(val_idx) if val_idx is not None else "N/A (missing or not loaded)")

print("\nPaths")
for k, p in required_paths.items():
    print(f"{k}: {p}")

Key Config Values
d_model: 256
FEWSHOT_N: 10
num_scenes: None
batch_size (from training_meta): 4096

Split Sizes
test_idx: 373033
val_idx: 373033

Paths
base_dir: /content/drive/MyDrive/Final_Year_Project/Implementation/datasets/AntM2C_10M
preprocessed_dir: /content/drive/MyDrive/Final_Year_Project/Implementation/datasets/AntM2C_10M/AntM2C_preprocessed
checkpoints_dir: /content/drive/MyDrive/Final_Year_Project/Implementation/datasets/AntM2C_10M/checkpoints
splits_dir: /content/drive/MyDrive/Final_Year_Project/Implementation/datasets/AntM2C_10M/AntM2C_preprocessed/splits
split_features_dir: /content/drive/MyDrive/Final_Year_Project/Implementation/datasets/AntM2C_10M/AntM2C_preprocessed/split_features
results_dir: /content/drive/MyDrive/Final_Year_Project/Implementation/datasets/AntM2C_10M/eval_results
model_ckpt_path: /content/drive/MyDrive/Final_Year_Project/Implementation/datasets/AntM2C_10M/checkpoints/coldfusionx_model_epoch3.pt
fusion_config_path: /content/drive/MyDrive/Final_Year_

### Load Row-Aligned Evaluation Artifacts + Assert Alignment

In [22]:
# Row-aligned artifacts (interaction-level)
df_id_path = os.path.join(split_features_dir, "df_id_encoded.pkl")
df_processed_path = os.path.join(preprocessed_dir, "df_processed.pkl")  # used for labels
text_embeddings_path = os.path.join(preprocessed_dir, "text_embeddings.npy")
time_features_path = os.path.join(preprocessed_dir, "time_features_final.pt")

In [23]:
# Item-level artifacts (lookup by item_id_idx)
image_embeddings_path = os.path.join(base_dir, "image_embeddings_vit_base_aligned.pt")
image_missing_mask_path = os.path.join(base_dir, "image_missing_mask.pt")

In [24]:
# Existence checks (fail early)
required_step1 = {
    "df_id_path": df_id_path,
    "df_processed_path": df_processed_path,
    "text_embeddings_path": text_embeddings_path,
    "time_features_path": time_features_path,
    "image_embeddings_path": image_embeddings_path,
    "image_missing_mask_path": image_missing_mask_path,
}

In [26]:
missing_step1 = [k for k, p in required_step1.items() if not os.path.exists(p)]
if missing_step1:
    print("Missing Step 1 artifacts:")
    for k in missing_step1:
        print(f" - {k}: {required_step1[k]}")
    raise FileNotFoundError("One or more Step 1 artifacts are missing. Fix paths before continuing.")

#### Load Artifacts (df_id, text memmap, time features, image tables, labels)

In [30]:
# Load df_id_encoded.pkl
df_id = pd.read_pickle(df_id_path)

In [31]:
# Load text embeddings (memmap)
text_memmap = np.load(text_embeddings_path, mmap_mode="r")

In [34]:
# Load time features
time_features = torch.load(time_features_path, map_location="cpu")

In [37]:
# Load item-level image embedding table + missing mask
image_embeddings = torch.load(image_embeddings_path, map_location="cpu")
image_missing_mask = torch.load(image_missing_mask_path, map_location="cpu")

In [38]:
# Load labels (same approach as Notebook 08: detect column)
df_processed = pd.read_pickle(df_processed_path)

In [58]:
!git commit -m "evaluation: revised setup and loading of relevant artifacts"